In [1]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MISSIONS = {
    "Wildfire Burn & Vegetation Stress (3U)": {
        "u_size": "3U CubeSat (~3.4 kg)",
        "alt_km": 520.0,
        "inc_deg": 97.5,
        "sensor": "Compact Dual-Band Red (660 nm) & NIR (850 nm) Imager",
        "gsd_m": 28.0,
        "swath_km": 42.0,
        "sample_mb_per_orbit": 65.0,
        "question": (
            "Can a 3U CubeSat in a 520 km sun-synchronous orbit detect post-fire"
            " vegetation recovery (ΔNDVI ≥ 0.15) across ≥ 75% of target burn"
            " scars within a 14-day revisit window?"
        ),
        "success_metric": "≥ 75% burn-scar coverage in ≤ 14 days at ≤ 30 m GSD with NDVI calibration error ≤ ±0.04",
    },
    "Inland & Coastal Water Turbidity (3U)": {
        "u_size": "3U CubeSat (~3.6 kg)",
        "alt_km": 500.0,
        "inc_deg": 53.0,
        "sensor": "Visible-SWIR Pushbroom Spectrometer (490–865 nm)",
        "gsd_m": 35.0,
        "swath_km": 55.0,
        "sample_mb_per_orbit": 80.0,
        "question": (
            "Can a 3U CubeSat resolve suspended sediment / turbidity shifts"
            " (≥ 15 NTU) in reservoirs and river plumes across seasonal runoff"
            " events with radiometric repeatability within ±8%?"
        ),
        "success_metric": "≥ 85% valid cloud-free water tiles per pass with turbidity RMSE ≤ 12 NTU",
    },
    "LEO Geomagnetic & Ionospheric Fluctuation (2U)": {
        "u_size": "2U CubeSat (~2.2 kg)",
        "alt_km": 480.0,
        "inc_deg": 82.0,
        "sensor": "3-Axis Fluxgate Magnetometer + Langmuir / GPS-RO Payload",
        "gsd_m": 0.0,
        "swath_km": 120.0,
        "sample_mb_per_orbit": 14.0,
        "question": (
            "Can a 2U CubeSat measure high-latitude/mid-latitude geomagnetic"
            " field perturbations (≥ 15 nT) correlated with solar-wind activity"
            " at 10 Hz sampling with noise floor ≤ 2 nT?"
        ),
        "success_metric": "≥ 90% continuous 10 Hz telemetry completeness over 30 days with ≤ 2 nT RMS noise",
    },
    "Surface Thermal Anomaly & Heat Island (6U)": {
        "u_size": "6U CubeSat (~9.5 kg)",
        "alt_km": 540.0,
        "inc_deg": 97.6,
        "sensor": "Uncooled LWIR Microbolometer (8–14 µm) + RGB Context Cam",
        "gsd_m": 45.0,
        "swath_km": 60.0,
        "sample_mb_per_orbit": 110.0,
        "question": (
            "Can a 6U CubeSat resolve daytime-to-nighttime land surface"
            " temperature contrasts (ΔT ≥ 4.0 K) between dense urban cores and"
            " surrounding rural/vegetated zones within ±1.5 K accuracy?"
        ),
        "success_metric": "Surface temperature accuracy ≤ ±1.5 K across ≥ 80% of scheduled target passes",
    },
}


def design_cubesat_mission(
    mission_key, passes_per_day, downlink_mb_per_pass, mission_days
):
  m = MISSIONS[mission_key]
  mu = 398600.4418
  re = 6378.137
  a_km = re + m["alt_km"]
  period_min = 2.0 * np.pi * np.sqrt((a_km**3) / mu) / 60.0
  orbits_per_day = (24.0 * 60.0) / period_min

  daily_gen_mb = orbits_per_day * m["sample_mb_per_orbit"]
  daily_down_mb = passes_per_day * downlink_mb_per_pass
  downlink_ratio_pct = min(100.0, (daily_down_mb / max(daily_gen_mb, 1e-6)) * 100.0)
  total_days = int(mission_days)
  backlog_mb = max(0.0, (daily_gen_mb - daily_down_mb) * total_days)

  days_arr = np.arange(1, total_days + 1)
  cum_gen_gb = (daily_gen_mb * days_arr) / 1024.0
  cum_down_gb = (np.minimum(daily_gen_mb, daily_down_mb) * days_arr) / 1024.0

  orbit_nums = np.arange(1, min(int(round(orbits_per_day * 3)) + 1, 50))
  # Illustrative coverage/quality score across orbits (0..100%)
  quality_score = np.clip(
      78.0
      + 12.0 * np.sin(orbit_nums * 0.45)
      - 6.0 * np.cos(orbit_nums * 0.22),
      55.0,
      98.5,
  )

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  ax1.plot(
      days_arr,
      cum_gen_gb,
      color="#38bdf8",
      linewidth=2.2,
      label="Payload Data Generated (GB)",
  )
  ax1.plot(
      days_arr,
      cum_down_gb,
      color="#34d399",
      linewidth=2.2,
      linestyle="--",
      label="Downlinked to Ground (GB)",
  )
  ax1.fill_between(
      days_arr, cum_down_gb, cum_gen_gb, color="#f87171", alpha=0.18, label="Onboard Buffer / Backlog"
  )
  ax1.set_title(
      f"Cumulative Data Budget ({total_days}-Day Window)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Mission Day", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Cumulative Data Volume (GB)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.grid(True, alpha=0.18, color="#64748b")
  ax1.legend(loc="upper left", fontsize=8, facecolor="#121826", edgecolor="#263248")
  ax1.set_facecolor("#111724")

  ax2.plot(
      orbit_nums,
      quality_score,
      color="#a78bfa",
      linewidth=2.0,
      marker="o",
      markersize=3.5,
      label="Observation Quality Index (%)",
  )
  ax2.axhline(
      75.0,
      color="#fbbf24",
      linestyle=":",
      linewidth=1.5,
      label="Success Threshold (75%)",
  )
  ax2.set_title(
      "Orbit-to-Orbit Measurement Quality Trend",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_xlabel("Orbit Index (First ~3 Days)", color="#94a3b8", fontsize=9)
  ax2.set_ylabel("Quality / Usability Score (%)", color="#94a3b8", fontsize=9)
  ax2.set_ylim(45, 105)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, alpha=0.18, color="#64748b")
  ax2.legend(loc="lower right", fontsize=8, facecolor="#121826", edgecolor="#263248")
  ax2.set_facecolor("#111724")

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">CUBESAT MISSION ARCHITECTURE • {m['u_size']}</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{mission_key} <span style="font-size:13px;color:#94a3b8;">({m['alt_km']:.0f} km • {m['inc_deg']:.1f}° inc)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Period: {period_min:.1f} min</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ORBITS / DAY</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">{orbits_per_day:.2f} rev/d</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DAILY DATA GENERATED</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{daily_gen_mb:.0f} MB/d</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DOWNLINK CAPACITY</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{downlink_ratio_pct:.1f}%</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">{total_days}-DAY BACKLOG</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{backlog_mb:.0f} MB</div>
        </div>
      </div>
    </div>
    """

  mission_box_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 20px;line-height:1.6;font-size:14px;color:#cbd5e1;">
      <div style="font-weight:700;font-size:15px;color:#f8fafc;margin-bottom:8px;">🛰️ Mission Core — Question, Sensor, Data Plan & Success Metric</div>
      <ul style="margin:0;padding-left:20px;">
        <li><b>Measurable Question:</b> {m['question']}</li>
        <li><b>Primary Sensor:</b> <b>{m['sensor']}</b> (swath ~{m['swath_km']:.0f} km, ~{m['sample_mb_per_orbit']:.0f} MB raw/compressed data per orbit).</li>
        <li><b>Data Plan:</b> Capture onboard during target passes → compress &amp; time-tag in flight computer → downlink across <b>{passes_per_day:.0f} ground passes/day</b> (~<b>{daily_down_mb:.0f} MB/day</b>) → calibrate, georeference, and validate against ground/reference datasets.</li>
        <li><b>Success Metric:</b> <b>{m['success_metric']}</b></li>
      </ul>
    </div>
    """

  plan_df = pd.DataFrame([
      {
          "Mission Element": "1. Measurable Question",
          "Specification": m["question"],
          "Verification Method": "Quantitative hypothesis test against target threshold",
      },
      {
          "Mission Element": "2. Payload / Sensor",
          "Specification": m["sensor"],
          "Verification Method": "Pre-flight calibration + in-orbit SNR / radiometric check",
      },
      {
          "Mission Element": "3. Data Plan",
          "Specification": f"{passes_per_day:.0f} passes/day × {downlink_mb_per_pass:.0f} MB/pass = {daily_down_mb:.0f} MB/day downlink vs {daily_gen_mb:.0f} MB/day generated",
          "Verification Method": "Onboard storage margin & packet loss rate log",
      },
      {
          "Mission Element": "4. Success Metric",
          "Specification": m["success_metric"],
          "Verification Method": "End-of-window requirement pass/fail scorecard",
      },
  ])

  return cards_html, fig, mission_box_html, plan_df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
.table-wrap, .dataframe {
  width: 100% !important;
  max-width: 100% !important;
  overflow-x: visible !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🛰️ CubeSat Mission Designer — Question, Sensor, Data Plan & Metric")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      mission_dd = gr.Dropdown(
          choices=list(MISSIONS.keys()),
          value=list(MISSIONS.keys())[0],
          label="CubeSat Mission Profile",
      )
      passes_sl = gr.Slider(
          minimum=1,
          maximum=8,
          value=4,
          step=1,
          label="Ground Station Passes / Day",
      )
      downlink_sl = gr.Slider(
          minimum=20,
          maximum=400,
          value=120,
          step=10,
          label="Downlink Volume / Pass (MB)",
      )
      days_sl = gr.Slider(
          minimum=7,
          maximum=60,
          value=14,
          step=1,
          label="Evaluation Window (Days)",
      )
      run_btn = gr.Button(
          "⚡ Generate CubeSat Mission Plan", variant="primary"
      )

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Data Budget & Orbit Quality Trends")

  with gr.Row():
    with gr.Column(scale=12):
      out_box = gr.HTML()
      out_table = gr.Dataframe(
          label="Mission Design & Verification Matrix", interactive=False
      )

  run_btn.click(
      fn=design_cubesat_mission,
      inputs=[mission_dd, passes_sl, downlink_sl, days_sl],
      outputs=[out_cards, out_plot, out_box, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [2]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MISSIONS = {
    "Wildfire Burn & Vegetation Stress (3U)": {
        "u_size": "3U CubeSat (~3.4 kg)",
        "alt_km": 520.0,
        "inc_deg": 97.5,
        "sensor": "Compact Dual-Band Red (660 nm) & NIR (850 nm) Imager",
        "gsd_m": 28.0,
        "swath_km": 42.0,
        "sample_mb_per_orbit": 65.0,
        "question": (
            "Can a 3U CubeSat at 520 km SSO detect post-fire vegetation"
            " recovery (ΔNDVI ≥ 0.15) across ≥ 75% of burn scars within 14"
            " days?"
        ),
        "short_q": (
            "Detect ΔNDVI ≥ 0.15 across ≥ 75% of burn scars within 14 days at"
            " 520 km SSO"
        ),
        "success_metric": (
            "≥ 75% burn-scar coverage in ≤ 14 days, ≤ 30 m GSD, NDVI error"
            " ≤ ±0.04"
        ),
    },
    "Inland & Coastal Water Turbidity (3U)": {
        "u_size": "3U CubeSat (~3.6 kg)",
        "alt_km": 500.0,
        "inc_deg": 53.0,
        "sensor": "Visible-SWIR Pushbroom Spectrometer (490–865 nm)",
        "gsd_m": 35.0,
        "swath_km": 55.0,
        "sample_mb_per_orbit": 80.0,
        "question": (
            "Can a 3U CubeSat resolve reservoir/plume turbidity shifts"
            " (≥ 15 NTU) across seasonal runoff events within ±8% repeatability?"
        ),
        "short_q": (
            "Measure turbidity shifts ≥ 15 NTU in reservoirs/plumes within ±8%"
            " repeatability"
        ),
        "success_metric": (
            "≥ 85% cloud-free water tiles per pass with turbidity RMSE ≤ 12"
            " NTU"
        ),
    },
    "LEO Geomagnetic & Ionospheric Fluctuation (2U)": {
        "u_size": "2U CubeSat (~2.2 kg)",
        "alt_km": 480.0,
        "inc_deg": 82.0,
        "sensor": "3-Axis Fluxgate Magnetometer + Langmuir / GPS-RO Payload",
        "gsd_m": 0.0,
        "swath_km": 120.0,
        "sample_mb_per_orbit": 14.0,
        "question": (
            "Can a 2U CubeSat measure geomagnetic perturbations (≥ 15 nT)"
            " correlated with solar-wind activity at 10 Hz with noise ≤ 2 nT?"
        ),
        "short_q": (
            "Measure geomagnetic shifts ≥ 15 nT at 10 Hz with noise floor ≤ 2"
            " nT"
        ),
        "success_metric": (
            "≥ 90% 10 Hz telemetry completeness over 30 days with ≤ 2 nT RMS"
            " noise"
        ),
    },
    "Surface Thermal Anomaly & Heat Island (6U)": {
        "u_size": "6U CubeSat (~9.5 kg)",
        "alt_km": 540.0,
        "inc_deg": 97.6,
        "sensor": "Uncooled LWIR Microbolometer (8–14 µm) + RGB Context Cam",
        "gsd_m": 45.0,
        "swath_km": 60.0,
        "sample_mb_per_orbit": 110.0,
        "question": (
            "Can a 6U CubeSat resolve day/night urban-rural surface ΔT ≥ 4.0 K"
            " within ±1.5 K accuracy?"
        ),
        "short_q": (
            "Resolve urban-rural surface ΔT ≥ 4.0 K within ±1.5 K accuracy"
        ),
        "success_metric": (
            "Surface temperature error ≤ ±1.5 K across ≥ 80% of target passes"
        ),
    },
}

DOWNLINK_STRATEGIES = {
    "ROI Filtering + Compression (Recommended)": {
        "retention_factor": 0.35,
        "desc": (
            "Discard cloud/off-target frames onboard, compress target ROIs"
            " (CCSDS/JPEG2000), and prioritize burn-scar/target tiles."
        ),
    },
    "Raw Full-Swath Downlink": {
        "retention_factor": 1.00,
        "desc": (
            "Downlink full generated frames without onboard ROI pruning;"
            " requires high pass count or large buffer."
        ),
    },
    "Moderate Compression Only": {
        "retention_factor": 0.65,
        "desc": (
            "Apply onboard compression without dropping non-target frames;"
            " reduces volume moderately."
        ),
    },
}


def design_cubesat_mission(
    mission_key, passes_per_day, downlink_mb_per_pass, mission_days, strategy_key
):
  m = MISSIONS[mission_key]
  st = DOWNLINK_STRATEGIES[strategy_key]
  mu = 398600.4418
  re = 6378.137
  a_km = re + m["alt_km"]
  period_min = 2.0 * np.pi * np.sqrt((a_km**3) / mu) / 60.0
  orbits_per_day = (24.0 * 60.0) / period_min

  raw_daily_mb = orbits_per_day * m["sample_mb_per_orbit"]
  effective_daily_mb = raw_daily_mb * st["retention_factor"]
  daily_down_mb = passes_per_day * downlink_mb_per_pass
  downlink_ratio_pct = min(
      100.0, (daily_down_mb / max(effective_daily_mb, 1e-6)) * 100.0
  )
  total_days = int(mission_days)
  backlog_mb = max(0.0, (effective_daily_mb - daily_down_mb) * total_days)

  days_arr = np.arange(1, total_days + 1)
  cum_eff_gb = (effective_daily_mb * days_arr) / 1024.0
  cum_down_gb = (np.minimum(effective_daily_mb, daily_down_mb) * days_arr) / 1024.0

  orbit_nums = np.arange(1, min(int(round(orbits_per_day * 3)) + 1, 50))
  quality_score = np.clip(
      78.0
      + 12.0 * np.sin(orbit_nums * 0.45)
      - 6.0 * np.cos(orbit_nums * 0.22),
      55.0,
      98.5,
  )

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  ax1.plot(
      days_arr,
      cum_eff_gb,
      color="#38bdf8",
      linewidth=2.2,
      label="Target Payload Data (GB)",
  )
  ax1.plot(
      days_arr,
      cum_down_gb,
      color="#34d399",
      linewidth=2.2,
      linestyle="--",
      label="Downlinked to Ground (GB)",
  )
  ax1.fill_between(
      days_arr,
      cum_down_gb,
      cum_eff_gb,
      color="#f87171",
      alpha=0.18,
      label="Remaining Backlog",
  )
  ax1.set_title(
      f"Cumulative Data Budget ({total_days}-Day Window)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Mission Day", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Cumulative Data Volume (GB)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.grid(True, alpha=0.18, color="#64748b")
  ax1.legend(
      loc="upper left", fontsize=8, facecolor="#121826", edgecolor="#263248"
  )
  ax1.set_facecolor("#111724")

  ax2.plot(
      orbit_nums,
      quality_score,
      color="#a78bfa",
      linewidth=2.0,
      marker="o",
      markersize=3.5,
      label="Observation Quality Index (%)",
  )
  ax2.axhline(
      75.0,
      color="#fbbf24",
      linestyle=":",
      linewidth=1.5,
      label="Success Threshold (75%)",
  )
  ax2.set_title(
      "Orbit-to-Orbit Measurement Quality Trend",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_xlabel("Orbit Index (First ~3 Days)", color="#94a3b8", fontsize=9)
  ax2.set_ylabel("Quality / Usability Score (%)", color="#94a3b8", fontsize=9)
  ax2.set_ylim(45, 105)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, alpha=0.18, color="#64748b")
  ax2.legend(
      loc="lower right",
      fontsize=8,
      facecolor="#121826",
      edgecolor="#263248",
  )
  ax2.set_facecolor("#111724")

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">CUBESAT MISSION ARCHITECTURE • {m['u_size']}</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{mission_key} <span style="font-size:13px;color:#94a3b8;">({m['alt_km']:.0f} km • {m['inc_deg']:.1f}° inc)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Period: {period_min:.1f} min</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ORBITS / DAY</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">{orbits_per_day:.2f} rev/d</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">EFFECTIVE DAILY DATA</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{effective_daily_mb:.0f} MB/d</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DOWNLINK COVERAGE</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{downlink_ratio_pct:.1f}%</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">{total_days}-DAY BACKLOG</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{backlog_mb:.0f} MB</div>
        </div>
      </div>
    </div>
    """

  mission_box_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 20px;line-height:1.6;font-size:14px;color:#cbd5e1;">
      <div style="font-weight:700;font-size:15px;color:#f8fafc;margin-bottom:8px;">🛰️ Mission Core — Question, Sensor, Data Plan & Success Metric</div>
      <ul style="margin:0;padding-left:20px;">
        <li><b>Measurable Question:</b> {m['question']}</li>
        <li><b>Primary Sensor:</b> <b>{m['sensor']}</b> (swath ~{m['swath_km']:.0f} km, ~{m['sample_mb_per_orbit']:.0f} MB raw per orbit).</li>
        <li><b>Data Plan ({strategy_key}):</b> {st['desc']} Daily downlink capacity is <b>{daily_down_mb:.0f} MB/day</b> ({passes_per_day:.0f} passes × {downlink_mb_per_pass:.0f} MB) against <b>{effective_daily_mb:.0f} MB/day</b> target data (down from {raw_daily_mb:.0f} MB/day raw).</li>
        <li><b>Success Metric:</b> <b>{m['success_metric']}</b></li>
      </ul>
    </div>
    """

  plan_df = pd.DataFrame([
      {
          "Element": "1. Measurable Question",
          "Summary": m["short_q"],
          "Verification": "Quantitative hypothesis test",
      },
      {
          "Element": "2. Payload / Sensor",
          "Summary": f"{m['sensor']} (~{m['sample_mb_per_orbit']:.0f} MB/orbit)",
          "Verification": "Calibration & SNR check",
      },
      {
          "Element": "3. Data Plan",
          "Summary": f"{daily_down_mb:.0f} MB/d downlinked vs {effective_daily_mb:.0f} MB/d ({strategy_key})",
          "Verification": "Buffer & packet log",
      },
      {
          "Element": "4. Success Metric",
          "Summary": m["success_metric"],
          "Verification": "End-of-window pass/fail",
      },
  ])

  return cards_html, fig, mission_box_html, plan_df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
.table-wrap, .dataframe {
  width: 100% !important;
  max-width: 100% !important;
  overflow-x: visible !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🛰️ CubeSat Mission Designer — Question, Sensor, Data Plan & Metric")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      mission_dd = gr.Dropdown(
          choices=list(MISSIONS.keys()),
          value=list(MISSIONS.keys())[0],
          label="CubeSat Mission Profile",
      )
      strategy_dd = gr.Dropdown(
          choices=list(DOWNLINK_STRATEGIES.keys()),
          value=list(DOWNLINK_STRATEGIES.keys())[0],
          label="Onboard Data & Downlink Strategy",
      )
      passes_sl = gr.Slider(
          minimum=1,
          maximum=8,
          value=4,
          step=1,
          label="Ground Station Passes / Day",
      )
      downlink_sl = gr.Slider(
          minimum=20,
          maximum=400,
          value=120,
          step=10,
          label="Downlink Volume / Pass (MB)",
      )
      days_sl = gr.Slider(
          minimum=7,
          maximum=60,
          value=14,
          step=1,
          label="Evaluation Window (Days)",
      )
      run_btn = gr.Button(
          "⚡ Generate CubeSat Mission Plan", variant="primary"
      )

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Data Budget & Orbit Quality Trends")

  with gr.Row():
    with gr.Column(scale=12):
      out_box = gr.HTML()
      out_table = gr.Dataframe(
          label="Mission Design & Verification Matrix", interactive=False
      )

  run_btn.click(
      fn=design_cubesat_mission,
      inputs=[mission_dd, passes_sl, downlink_sl, days_sl, strategy_dd],
      outputs=[out_cards, out_plot, out_box, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>